# 04 · 最终模型与提交

最终模型 = **CatBoost（GPU）三组配置 × 3 个随机种子，全部在 105,000 行上训练**，测试概率先按组平均种子、再三组等权平均。公开榜 **0.92301**。

| 组 | 参数（03） | 特征（02） |
|---|---|---|
| M1 | C11：depth 6、lr 0.03、L2=10、Bernoulli 0.8 | G7（72 列） |
| M2 | C7：depth 8、lr 0.03、L2=10、Bernoulli 0.8 | G7 |
| M3 | C11 | LEAN（68 列） |
| M0（对照） | C11 | 第一轮全部特征（68 列） |

步骤：① 五折验证每一组与它们的融合；② 学习曲线——判断「更多训练数据」是否有价值；③ 在全部训练数据上重训并生成提交文件。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 原始数据读取、审计、探索分析、数据质量检查、固定五折 |
| **02_feature_engineering** | 全部特征的定义、三轮特征消融、保存特征库 |
| **03_model_selection** | 基线模型（逻辑回归、XGBoost、CatBoost、AutoGluon）、线性模型的数值变换、CatBoost 调参与训练方式筛选 |
| **04_final_model** | 最终五折模型、多种子、学习曲线、全量重训与最终提交 |
| **05_error_analysis** | 残差诊断、LightGBM 与神经网络、跨实验融合、伪标签、公开榜噪声与运行间差异 |

原始数据在 `data/`；每本 notebook 的结果保存在 `artifacts/` 下与它同名的目录。

In [ ]:
# 环境、路径与通用函数（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def find_project_dir():
    candidates = []
    if os.environ.get('HMDA_PROJECT_DIR'):
        candidates.append(Path(os.environ['HMDA_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 HMDA_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                                        # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'                                  # 每本 notebook 的结果写入同名子目录
PREPARED_DIR = ARTIFACTS / '01_data_exploration'                       # 01：固定数据接口
FEATURE_DIR = ARTIFACTS / '02_feature_engineering' / 'feature_store'   # 02：特征库
SEED = 42
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


def ensure_catboost():
    """只在训练时调用：安装并导入固定版本的 CatBoost。"""
    global catboost
    try:
        installed = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已导入另一版 CatBoost；请重启运行时后从头运行')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'catboost=={CATBOOST_VERSION}'])
        importlib.invalidate_caches()
    import catboost as catboost_module
    if catboost_module.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}，实际为 {catboost_module.__version__}')
    catboost = catboost_module
    return catboost_module


def ensure_gpu():
    """只在训练时调用：确认 Colab 分配了 NVIDIA GPU。"""
    global GPU_NAME
    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('训练需要 GPU：Colab「更改运行时类型」选 T4 GPU 后从头运行')
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           text=True, capture_output=True, check=False)
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError('NVIDIA GPU 不可用：请选择 T4 GPU 后从头运行')
    GPU_NAME = probe.stdout.strip().splitlines()[0]
    return GPU_NAME


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

**训练开关**：默认全部为 `False`，只读取 `artifacts/` 中已保存的结果并生成表格和图（Colab 普通 CPU 运行时即可，几分钟跑完）。把某一项改为 `True` 并选择 **T4 GPU** 运行时，会按原协议重新训练对应部分；每个模型训练完立即写入 `artifacts/`，断线后从头运行会跳过已完成的部分。

In [ ]:
TRAIN = {
    'cv_models': False,        # 五折模型：M0 与 M1–M3 × 3 个种子（GPU，约 6 小时）
    'learning_curve': False,   # 学习曲线：10 次训练（GPU，约 75 分钟）
    'full_refit': False,       # 全量重训：9 个模型（GPU，约 50 分钟）
}

## 0. 读取固定数据接口与特征库

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_data_exploration.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = DATA_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    DATA_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    DATA_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

In [ ]:
# 读取 02 生成的特征库（只在重新训练时真正需要；只读结果时缺少也不影响）
def load_feature_store():
    meta_path = FEATURE_DIR / 'feature_store.json'
    require(meta_path.is_file(), '缺少特征库：请先运行 02_feature_engineering.ipynb')
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    require(meta['data_hashes'] == DATA_HASHES, '特征库与 01 的固定数据接口不一致：请重跑 02')
    stores = {}
    for part, expected_rows in [('train', len(X_base)), ('test', len(X_test_base))]:
        path = FEATURE_DIR / f'blocks_{part}.parquet'
        require(sha256_file(path) == meta['files'][path.name], f'{path.name} 哈希不符：请重跑 02')
        frame = pd.read_parquet(path)
        require(len(frame) == expected_rows, f'{path.name} 行数不符')
        for column in frame.columns:
            frame[column] = frame[column].astype(object if column in meta['categorical'] else 'float64')
        stores[part] = frame
    blocks = {name: (stores['train'][cols].reset_index(drop=True), stores['test'][cols].reset_index(drop=True))
              for name, cols in meta['blocks'].items()}
    return blocks, set(meta['categorical']), meta


def variant_frames(blocks):
    """原始 56 列 + 指定的特征块；与 02 中的定义相同。"""
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


if (FEATURE_DIR / 'feature_store.json').is_file():
    BLOCKS, NEW_CATEGORICAL, FEATURE_META = load_feature_store()
    print(f"特征库：{len(BLOCKS)} 个特征块、{sum(len(c) for c in FEATURE_META['blocks'].values())} 列；"
          f"特征集 {list(FEATURE_META['feature_sets'])}")
else:
    BLOCKS, NEW_CATEGORICAL, FEATURE_META = None, set(), None
    print('尚未生成特征库（02）：只读取已保存结果不受影响；需要重新训练时请先运行 02。')

In [ ]:
# 只读的结果工具：读取每个（模型/变体, 折）保存的 JSON，做同折配对比较
def load_runs(directory, pattern='*__fold?.json'):
    directory = Path(directory)
    paths = sorted(directory.glob(pattern))
    require(paths, f'{directory} 中没有结果文件：请确认 artifacts 已从 Drive 同步，或打开对应的训练开关')
    return pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in paths])


def paired_table(runs, key, baseline, min_gain=0.0005, min_folds=4):
    auc = runs.pivot(index='fold', columns=key, values='valid_auc')
    require(baseline in auc.columns, f'缺少基线 {baseline}')
    delta = auc.sub(auc[baseline], axis=0)
    table = pd.DataFrame({
        'folds': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'delta_vs_base': delta.mean(),
        'folds_better': (delta > 0).sum(),
    })
    table['keep'] = ((table['folds'] == 5) & (table['delta_vs_base'] >= min_gain)
                     & (table['folds_better'] >= min_folds))
    table.loc[baseline, 'keep'] = False
    return table.sort_values('mean_auc', ascending=False), delta


def plot_deltas(delta, baseline, title):
    others = delta.drop(columns=[baseline])
    order = others.mean().sort_values().index.tolist()
    means = others[order].mean() * 1e4
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(order) + 1.6))
    ax.barh(range(len(order)), means, color=['#4c72b0' if v > 0 else '#c44e52' for v in means])
    for i, name in enumerate(order):
        ax.scatter(others[name] * 1e4, np.full(len(others), i), color='black', s=12, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(range(len(order)), order)
    ax.set_xlabel(f'AUC change vs {baseline} (x1e-4); bar = 5-fold mean, dots = single folds')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()



# 读取五折验证预测（OOF）：每个文件是一个外层折的验证行
def read_oof(directory, group):
    frames = [pd.read_csv(Path(directory) / f'{group}__fold{k}__valid.csv').assign(fold=k) for k in range(1, 6)]
    return pd.concat(frames).sort_values('id', kind='stable').reset_index(drop=True)


def fold_auc(frame, probability):
    probability = np.asarray(probability)
    return pd.Series({k: roc_auc_score(frame.loc[frame['fold'].eq(k), 'accepted'],
                                       probability[frame['fold'].eq(k).to_numpy()]) for k in range(1, 6)})


def mean_test(paths):
    frames = [pd.read_csv(p, dtype={'id': str}) for p in paths]
    for f in frames:
        require(f['id'].equals(sample_submission['id']), f'测试 ID 顺序不一致：{Path(paths[0]).name}')
    return np.mean([f['accepted'].to_numpy() for f in frames], axis=0)


CV_DIR = ARTIFACTS / '04_final_model' / 'cv_models'      # 五折模型（M0 与 M1–M3 × 3 个种子）
FULL_DIR = ARTIFACTS / '04_final_model' / 'full_refit'   # 学习曲线与全量模型
CAT_BASE_GROUPS = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
SEED_TAGS = {42: '', 2027: '_s2027', 31415: '_s31415'}

## 1. 五折模型与融合

**协议**：每个外层折的训练部分内留出 15% 早停，得到最佳轮数 *b*，再在整个外层训练部分按 `round(b / 0.85)` 轮重训；外层验证只评分。每个折模型同时预测测试集。M1/M2/M3 各训练 3 个种子（42、2027、31415）。

In [ ]:
EXPERIMENT_ID = 'cv_models'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None             # None：训练所有缺失的组；已完成的会自动跳过
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = ARTIFACTS / '04_final_model' / EXPERIMENT_ID

GPU_BASE = dict(
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    allow_writing_files=False, verbose=False,
)
ALL_BLOCKS = ['dti', 'ltv', 'income', 'missing', 'freq']
NO_FREQ = ['dti', 'ltv', 'income', 'missing']
FAST = dict(depth=8, learning_rate=0.05, l2_leaf_reg=3,
            bootstrap_type='Bernoulli', subsample=0.8, early_stop=False, iterations=1600)
C7 = dict(depth=8, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
C11 = dict(depth=6, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
ROUND2 = ['precision', 'thresholds', 'implied', 'term']
G7 = NO_FREQ + ROUND2                      # 02 特征工程中最好的特征集（72 列）
LEAN = NO_FREQ + ['precision', 'thresholds']  # 第二轮只保留单独有效的两块（68 列）
GROUPS = {
    'M0_C11_all': dict(params=C11, blocks=ALL_BLOCKS),   # 第一轮全部特征（公开榜 0.92073 的配置），同轮对照
    'M1_C11_G7': dict(params=C11, blocks=G7),
    'M2_C7_G7': dict(params=C7, blocks=G7),
    'M3_C11_lean': dict(params=C11, blocks=LEAN),
}
BASELINE = 'M0_C11_all'
CAT_BASES = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
SEEDS_EXTRA = [2027, 31415]          # 额外的两个模型种子（基础组用 seed 42）
for base_name in CAT_BASES:
    for extra_seed in SEEDS_EXTRA:
        base_group = GROUPS[base_name]
        GROUPS[f'{base_name}_s{extra_seed}'] = dict(
            params=dict(base_group['params'], random_seed=extra_seed), blocks=base_group['blocks'])

if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(GROUPS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知组：{sorted(unknown)}')

In [ ]:
if TRAIN['cv_models']:
    ensure_catboost()
    ensure_gpu()
    fixed_config(OUTPUT_DIR, {
        'purpose': 'params x feature blocks, 5-fold with inner early stop; fold-averaged test predictions',
        'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
        'max_iterations': MAX_ITERATIONS, 'od_wait': OD_WAIT,
        'gpu_base': {k: v for k, v in GPU_BASE.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })


    def model_params(spec, iterations):
        params = {k: v for k, v in spec.items() if k not in {'early_stop', 'iterations'}}
        return dict(GPU_BASE, **params, iterations=iterations)


    def run_one(name, group, fold):
        spec = group['params']
        train, test, cats = variant_frames(group['blocks'])
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        started = time.perf_counter()
        best_iteration, early_stopped = None, None
        if spec.get('early_stop', True):
            inner_fit, inner_stop = train_test_split(
                fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED,
                stratify=y.iloc[fit_idx],
            )
            probe = catboost.CatBoostClassifier(**model_params(spec, MAX_ITERATIONS),
                                                eval_metric='Logloss', od_type='Iter',
                                                od_wait=OD_WAIT, use_best_model=True)
            probe.fit(train.iloc[inner_fit], y.iloc[inner_fit], cat_features=cats,
                      eval_set=(train.iloc[inner_stop], y.iloc[inner_stop]))
            best_iteration = int(probe.get_best_iteration()) + 1
            early_stopped = int(probe.tree_count_) < MAX_ITERATIONS
            if not early_stopped:
                print(f'  注意：{name} 到 {MAX_ITERATIONS} 轮仍未早停')
            rounds = int(round(best_iteration / (1 - INNER_STOP_FRACTION)))
            del probe
            gc.collect()
        else:
            rounds = int(spec['iterations'])
        model = catboost.CatBoostClassifier(**model_params(spec, rounds), use_best_model=False)
        model.fit(train.iloc[fit_idx], y.iloc[fit_idx], cat_features=cats)
        require(int(model.tree_count_) == rounds, '外层重训轮数不符')
        valid_prob = checked_probabilities(model, train.iloc[valid_idx])
        test_prob = checked_probabilities(model, test)
        actual = y.iloc[valid_idx]
        stem = OUTPUT_DIR / f'{name}__fold{fold}'
        pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                      'prob': valid_prob}).to_csv(f'{stem}__valid.csv', index=False, float_format='%.12f')
        pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': test_prob}).to_csv(
            f'{stem}__test.csv', index=False, float_format='%.12f')
        result = {
            'group': name, 'fold': fold, 'definition': group,
            'feature_count': train.shape[1],
            'inner_best_iteration': best_iteration, 'early_stopped': early_stopped,
            'refit_rounds': rounds,
            'valid_auc': roc_auc_score(actual, valid_prob),
            'valid_logloss': log_loss(actual, valid_prob, labels=[0, 1]),
            'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
            'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        write_json(Path(f'{stem}.json'), result)   # 最后写 json，作为该折完成的标志
        del model, train, test
        gc.collect()
        return result


    selected = {n: g for n, g in GROUPS.items() if RUN_ONLY is None or n in RUN_ONLY}
    for fold in FOLDS:
        for name, group in selected.items():
            result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if result_path.is_file():
                saved = json.loads(result_path.read_text(encoding='utf-8'))
                require(saved['definition'] == group, f'{name} 的定义已改变；请改组名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            result = run_one(name, group, fold)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
                  f'轮数={result["refit_rounds"]}  {result["seconds"]/60:.1f} 分钟', flush=True)

    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                            for p in sorted(OUTPUT_DIR.glob('*__fold?.json'))])
    done = {g for g, part in results.groupby('group') if sorted(part['fold']) == [1, 2, 3, 4, 5]}
    print('五折完成的组：', sorted(done))


    def load(name):
        valid = pd.concat([pd.read_csv(OUTPUT_DIR / f'{name}__fold{k}__valid.csv').assign(fold=k)
                           for k in range(1, 6)]).sort_values('id', kind='stable').reset_index(drop=True)
        tests = [pd.read_csv(OUTPUT_DIR / f'{name}__fold{k}__test.csv', dtype={'id': str}) for k in range(1, 6)]
        for frame in tests:
            require(frame['id'].equals(sample_submission['id']), f'{name} 测试 ID 顺序不一致')
        return valid, np.mean([t['accepted'].to_numpy() for t in tests], axis=0)


    loaded = {name: load(name) for name in sorted(done)}
    reference = loaded[CAT_BASES[0]][0][['id', 'accepted', 'fold']]
    for name, (valid, _) in loaded.items():
        require(valid['id'].equals(reference['id']), f'{name} OOF 的 ID 与参照不一致')


    def mean_of(names):
        return (np.mean([loaded[n][0]['prob'].to_numpy() for n in names], axis=0),
                np.mean([loaded[n][1] for n in names], axis=0))


    def fold_aucs(probability):
        return pd.Series({k: roc_auc_score(reference.loc[reference['fold'].eq(k), 'accepted'],
                                           probability[reference['fold'].eq(k).to_numpy()])
                          for k in range(1, 6)})


    blends = {}
    require(all(b in done for b in CAT_BASES), '缺少 M1–M3 的五折结果')
    blends['B13_seed42'] = mean_of(CAT_BASES)                       # = 已提交的 0.92190
    per_base = [mean_of([b] + [f'{b}_s{s}' for s in SEEDS_EXTRA if f'{b}_s{s}' in done]) for b in CAT_BASES]
    n_seeds = {b: 1 + sum(f'{b}_s{s}' in done for s in SEEDS_EXTRA) for b in CAT_BASES}
    seed_tag = 'B_seeds_' + '_'.join(str(n_seeds[b]) for b in CAT_BASES)
    blends[seed_tag] = (np.mean([v for v, _ in per_base], axis=0), np.mean([t for _, t in per_base], axis=0))

    ref_folds = fold_aucs(blends['B13_seed42'][0])
    rows = []
    for name, (valid_prob, test_prob) in blends.items():
        folds = fold_aucs(valid_prob)
        require(np.isfinite(test_prob).all() and len(test_prob) == 45000, f'{name} 测试概率无效')
        path = OUTPUT_DIR / f'submission_{name}.csv'
        pd.DataFrame({'id': sample_submission['id'], 'accepted': test_prob}).to_csv(
            path, index=False, float_format='%.12f')
        rows.append({'blend': name, 'mean_auc': folds.mean(),
                     'delta_vs_B13': (folds - ref_folds).mean(),
                     'folds_better_than_B13': int(((folds - ref_folds) > 0).sum()),
                     'oof_auc': roc_auc_score(reference['accepted'], valid_prob),
                     'submission': path.name})
    table = pd.DataFrame(rows).set_index('blend').sort_values('mean_auc', ascending=False)
    table.to_csv(OUTPUT_DIR / 'blend_summary.csv')
    display(table.round(6))
    singles = results.groupby('group')['valid_auc'].agg(['count', 'mean']).round(6)
    display(singles.sort_values('mean', ascending=False))
    best = table['mean_auc'].idxmax()
    print(f'五折平均最高：{best}（{table.loc[best, "mean_auc"]:.6f}，相对已提交 {table.loc[best, "delta_vs_B13"]:+.6f}）'
          f' → {table.loc[best, "submission"]}')
else:
    print("训练开关关闭（TRAIN['cv_models'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
cv_runs = load_runs(CV_DIR)
per_group = cv_runs.pivot(index='fold', columns='group', values='valid_auc')
display(per_group.agg(['mean', 'std']).T.sort_values('mean', ascending=False).round(6))

oof = {g: read_oof(CV_DIR, g) for g in ['M0_C11_all'] + [b + t for b in CAT_BASE_GROUPS for t in SEED_TAGS.values()]}
reference = oof['M0_C11_all']
for name, frame in oof.items():
    require(frame['id'].equals(reference['id']) and frame['fold'].equals(reference['fold']), f'{name} 的 OOF 未对齐')
blend_seed42 = np.mean([oof[b]['prob'] for b in CAT_BASE_GROUPS], axis=0)
b_seeds = np.mean([np.mean([oof[b + t]['prob'] for t in SEED_TAGS.values()], axis=0) for b in CAT_BASE_GROUPS], axis=0)
candidates = {
    'M0: C11 + round-1 features': reference['prob'],
    'M3: C11 + LEAN': oof['M3_C11_lean']['prob'],
    'M1+M2+M3 (seed 42)': blend_seed42,
    'B_seeds: M1/M2/M3 x 3 seeds': b_seeds,
}
base_folds = fold_auc(reference, reference['prob'])
fold_table = pd.DataFrame({name: fold_auc(reference, prob) for name, prob in candidates.items()})
summary = pd.DataFrame({'五折 AUC': fold_table.mean(), '相对 M0': fold_table.sub(base_folds, axis=0).mean(),
                        '更好折数': fold_table.sub(base_folds, axis=0).gt(0).sum()})
display(summary.round(6))

fig, ax = plt.subplots(figsize=(7, 3.8))
for name in fold_table.columns:
    ax.plot(fold_table.index, fold_table[name], marker='o', label=name)
ax.set_xticks(range(1, 6))
ax.set_xlabel('fold')
ax.set_ylabel('validation AUC')
ax.set_title('Final 5-fold models')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**结论**：第二轮特征让单组提升 +0.0006～+0.0009；三组融合 0.92051；每组 3 个种子再平均 **0.92073**（相对 M0 +0.0014，4/5 折）。这是能直接用五折验证的最好结果。

## 2. 学习曲线：更多训练数据值多少？

上面的提交方式是「五个折模型的测试预测平均」，每个模型只见过 80% 的训练数据（84,000 行）。在同一折上，M3 分别用外层训练部分的 80%（67,200 行）和 100%（84,000 行）训练，比较验证 AUC。67k→84k 与 84k→105k 都是 ×1.25，前者的收益就是全量重训收益的估计。

In [ ]:
EXPERIMENT_ID = 'full_refit'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = ARTIFACTS / '04_final_model' / EXPERIMENT_ID

GPU_BASE = dict(
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    allow_writing_files=False, verbose=False,
)
ALL_BLOCKS = ['dti', 'ltv', 'income', 'missing', 'freq']
NO_FREQ = ['dti', 'ltv', 'income', 'missing']
FAST = dict(depth=8, learning_rate=0.05, l2_leaf_reg=3,
            bootstrap_type='Bernoulli', subsample=0.8, early_stop=False, iterations=1600)
C7 = dict(depth=8, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
C11 = dict(depth=6, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
ROUND2 = ['precision', 'thresholds', 'implied', 'term']
G7 = NO_FREQ + ROUND2                      # 02 特征工程中最好的特征集（72 列）
LEAN = NO_FREQ + ['precision', 'thresholds']  # 第二轮只保留单独有效的两块（68 列）
GROUPS = {
    'M0_C11_all': dict(params=C11, blocks=ALL_BLOCKS),   # 第一轮全部特征（公开榜 0.92073 的配置），同轮对照
    'M1_C11_G7': dict(params=C11, blocks=G7),
    'M2_C7_G7': dict(params=C7, blocks=G7),
    'M3_C11_lean': dict(params=C11, blocks=LEAN),
}
BASELINE = 'M0_C11_all'
CATBOOST_DIR = ARTIFACTS / '04_final_model' / 'cv_models'   # 第 1 节的五折模型结果
CAT_BASES = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
SEED_SUFFIXES = {42: '', 2027: '_s2027', 31415: '_s31415'}
LC_GROUP = 'M3_C11_lean'       # 学习曲线只用五折最好的单组
LC_FRACTION = 0.8              # 外层训练部分的 80%（67,200 行）对比 100%（84,000 行）
FULL_ROUNDS_SCALE = 105000 / 84000   # 全量轮数 = 该组五折重训轮数均值 × 数据量之比（与 b/0.85 同一规则）
RUN_LEARNING_CURVE = TRAIN['learning_curve']
RUN_FULL_REFIT = TRAIN['full_refit']
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(GROUPS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知组：{sorted(unknown)}')

In [ ]:
if TRAIN['learning_curve'] or TRAIN['full_refit']:
    ensure_catboost()
    ensure_gpu()
    fixed_config(OUTPUT_DIR, {
        'purpose': 'learning curve (80% vs 100% of outer-train) + full-data refit of M1-M3 x 3 seeds',
        'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
        'max_iterations': MAX_ITERATIONS, 'od_wait': OD_WAIT,
        'lc_group': LC_GROUP, 'lc_fraction': LC_FRACTION, 'full_rounds_scale': FULL_ROUNDS_SCALE,
        'gpu_base': {k: v for k, v in GPU_BASE.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })


    def model_params(spec, iterations, seed):
        params = {k: v for k, v in spec.items() if k not in {'early_stop', 'iterations'}}
        return dict(GPU_BASE, **params, iterations=iterations, random_seed=seed)


    def early_stop_refit(spec, train, cats, idx, seed):
        """与五折模型（04）完全相同的协议：idx 内留 15% 早停 → 在整个 idx 上按 b/0.85 轮重训。"""
        inner_fit, inner_stop = train_test_split(idx, test_size=INNER_STOP_FRACTION,
                                                 random_state=SEED, stratify=y.iloc[idx])
        probe = catboost.CatBoostClassifier(**model_params(spec, MAX_ITERATIONS, seed),
                                            eval_metric='Logloss', od_type='Iter',
                                            od_wait=OD_WAIT, use_best_model=True)
        probe.fit(train.iloc[inner_fit], y.iloc[inner_fit], cat_features=cats,
                  eval_set=(train.iloc[inner_stop], y.iloc[inner_stop]))
        best = int(probe.get_best_iteration()) + 1
        rounds = int(round(best / (1 - INNER_STOP_FRACTION)))
        del probe
        gc.collect()
        model = catboost.CatBoostClassifier(**model_params(spec, rounds, seed), use_best_model=False)
        model.fit(train.iloc[idx], y.iloc[idx], cat_features=cats)
        require(int(model.tree_count_) == rounds, '重训轮数不符')
        return model, best, rounds
else:
    print("训练开关关闭（TRAIN['learning_curve'] or TRAIN['full_refit'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
if TRAIN['learning_curve']:
    ensure_catboost()
    ensure_gpu()
    if RUN_LEARNING_CURVE:
        lc_spec = GROUPS[LC_GROUP]
        lc_train, _, lc_cats = variant_frames(lc_spec['blocks'])
        for fold in [1, 2, 3, 4, 5]:
            fit_idx = np.flatnonzero(fold_assignment != fold)
            valid_idx = np.flatnonzero(fold_assignment == fold)
            sub_idx, _ = train_test_split(fit_idx, train_size=LC_FRACTION, random_state=SEED + fold,
                                          stratify=y.iloc[fit_idx])
            for name, idx in [('LC_sub80', np.sort(sub_idx)), ('LC_full', fit_idx)]:
                path = OUTPUT_DIR / f'{name}__fold{fold}.json'
                if path.is_file():
                    saved = json.loads(path.read_text(encoding='utf-8'))
                    print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                    continue
                started = time.perf_counter()
                model, best, rounds = early_stop_refit(lc_spec['params'], lc_train, lc_cats, idx, SEED)
                probability = checked_probabilities(model, lc_train.iloc[valid_idx])
                result = {'name': name, 'fold': fold, 'group': LC_GROUP, 'train_rows': int(len(idx)),
                          'inner_best_iteration': best, 'refit_rounds': rounds,
                          'valid_auc': roc_auc_score(y.iloc[valid_idx], probability),
                          'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                          'finished_at_utc': datetime.now(timezone.utc).isoformat()}
                write_json(path, result)
                print(f'{name} fold {fold}: {len(idx):,} 行  AUC={result["valid_auc"]:.6f}  '
                      f'轮数={rounds}  {result["seconds"]/60:.1f} 分钟', flush=True)
                del model
                gc.collect()

    lc = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in OUTPUT_DIR.glob('LC_*__fold?.json')])
    if not lc.empty:
        curve = lc.pivot(index='fold', columns='name', values='valid_auc')
        if {'LC_sub80', 'LC_full'} <= set(curve.columns):
            curve['gain_84k_vs_67k'] = curve['LC_full'] - curve['LC_sub80']
        reference = {k: json.loads((CATBOOST_DIR / f'{LC_GROUP}__fold{k}.json').read_text(encoding='utf-8'))['valid_auc']
                     for k in curve.index}
        curve['M3_in_13'] = pd.Series(reference)
        display(curve.round(6))
        if 'gain_84k_vs_67k' in curve:
            done = curve['gain_84k_vs_67k'].dropna()
            print(f'训练数据 ×1.25（67k → 84k）：平均 AUC {done.mean():+.6f}，{int((done > 0).sum())}/{len(done)} 折为正。'
                  f' 全量重训（84k → 105k）是同样的 ×1.25，预期收益与此同量级。')
else:
    print("训练开关关闭（TRAIN['learning_curve'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
lc = load_runs(FULL_DIR, 'LC_*__fold?.json')
curve = lc.pivot(index='fold', columns='name', values='valid_auc')
curve['gain_84k_vs_67k'] = curve['LC_full'] - curve['LC_sub80']
display(curve.round(6))
print(f"训练数据 ×1.25（67,200 → 84,000 行）：平均 AUC {curve['gain_84k_vs_67k'].mean():+.5f}，"
      f"{int((curve['gain_84k_vs_67k'] > 0).sum())}/5 折为正")

fig, ax = plt.subplots(figsize=(6, 3.8))
for fold, row in curve.iterrows():
    ax.plot([67200, 84000], [row['LC_sub80'], row['LC_full']], marker='o', label=f'fold {fold}')
ax.set_xlabel('training rows')
ax.set_ylabel('validation AUC')
ax.set_title('Learning curve: M3, 80% vs 100% of outer-train')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**结论**：训练数据 ×1.25 → AUC **+0.0012，5/5 折为正**。模型受数据量限制，用上全部 105,000 行是值得的。

## 3. 全量重训与提交文件

M1/M2/M3 × 3 个种子在全部 105,000 行上训练，轮数 = 该组五折重训轮数均值 × 1.25（与 `b / 0.85` 同一规则）。全量模型没有留出的验证集，是否采用只依据第 2 节的学习曲线。下面重新计算两份提交文件，并与保存的文件核对。

In [ ]:
if TRAIN['full_refit']:
    ensure_catboost()
    ensure_gpu()
    def group_rounds(group):
        rounds = [json.loads((CATBOOST_DIR / f'{group}__fold{k}.json').read_text(encoding='utf-8'))['refit_rounds']
                  for k in range(1, 6)]
        return float(np.mean(rounds))


    if RUN_FULL_REFIT:
        for base in CAT_BASES:
            spec = GROUPS[base]
            train, test, cats = variant_frames(spec['blocks'])
            for seed, suffix in SEED_SUFFIXES.items():
                name = f'FULL_{base}_seed{seed}'
                path = OUTPUT_DIR / f'{name}.json'
                if path.is_file():
                    print(f'{name}: 已完成，跳过')
                    continue
                rounds = int(round(group_rounds(base + suffix) * FULL_ROUNDS_SCALE))
                started = time.perf_counter()
                model = catboost.CatBoostClassifier(**model_params(spec['params'], rounds, seed),
                                                    use_best_model=False)
                model.fit(train, y, cat_features=cats)
                require(int(model.tree_count_) == rounds, '全量模型轮数不符')
                probability = checked_probabilities(model, test)
                pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': probability}).to_csv(
                    OUTPUT_DIR / f'{name}__test.csv', index=False, float_format='%.12f')
                write_json(path, {'name': name, 'base': base, 'seed': seed, 'train_rows': len(train),
                                  'rounds': rounds, 'seconds': time.perf_counter() - started,
                                  'gpu_name': GPU_NAME,
                                  'finished_at_utc': datetime.now(timezone.utc).isoformat()})
                print(f'{name}: {rounds} 轮  {(time.perf_counter() - started)/60:.1f} 分钟', flush=True)
                del model
                gc.collect()

    def fold_average_bseeds():
        per_base = []
        for base in CAT_BASES:
            tests = [pd.read_csv(CATBOOST_DIR / f'{base}{s}__fold{k}__test.csv', dtype={'id': str})
                     for s in SEED_SUFFIXES.values() for k in range(1, 6)]
            for t in tests:
                require(t['id'].equals(sample_submission['id']), '折模型测试 ID 顺序不一致')
            per_base.append(np.mean([t['accepted'].to_numpy() for t in tests], axis=0))
        return np.mean(per_base, axis=0)


    full_paths = {(b, s): OUTPUT_DIR / f'FULL_{b}_seed{s}__test.csv' for b in CAT_BASES for s in SEED_SUFFIXES}
    missing = [p.name for p in full_paths.values() if not p.is_file()]
    if missing:
        print(f'全量模型还缺 {len(missing)} 个（例如 {missing[0]}）；完成后再运行本格。')
    else:
        per_base = []
        for base in CAT_BASES:
            frames = [pd.read_csv(full_paths[(base, s)], dtype={'id': str}) for s in SEED_SUFFIXES]
            for f in frames:
                require(f['id'].equals(sample_submission['id']), '全量模型测试 ID 顺序不一致')
            per_base.append(np.mean([f['accepted'].to_numpy() for f in frames], axis=0))
        full = np.mean(per_base, axis=0)
        fold_avg = fold_average_bseeds()
        print(f'全量融合 与 折平均 B_seeds 的测试预测相关系数：{np.corrcoef(full, fold_avg)[0, 1]:.5f}')
        for name, probability in [('full_Bseeds', full), ('hybrid_50_50', 0.5 * full + 0.5 * fold_avg)]:
            require(np.isfinite(probability).all() and len(probability) == 45000, f'{name} 概率无效')
            path = OUTPUT_DIR / f'submission_{name}.csv'
            pd.DataFrame({'id': sample_submission['id'], 'accepted': probability}).to_csv(
                path, index=False, float_format='%.12f')
            print('写出', path.name)
        if 'curve' in globals() and 'gain_84k_vs_67k' in curve and curve['gain_84k_vs_67k'].notna().sum() == 5:
            gain = curve['gain_84k_vs_67k']
            if gain.mean() >= 0.0005 and (gain > 0).sum() >= 4:
                advice = '学习曲线支持全量重训：最终主选 submission_hybrid_50_50.csv，备选保留折平均的 B_seeds。'
            elif gain.mean() >= 0.0002:
                advice = '学习曲线收益偏小：hybrid 可以作为备选，主选仍为折平均的 B_seeds。'
            else:
                advice = '学习曲线没有显示数据量收益：保持折平均的 B_seeds，不使用全量模型。'
            print('判断：' + advice)
else:
    print("训练开关关闭（TRAIN['full_refit'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
full_runs = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in sorted(FULL_DIR.glob('FULL_*.json'))])
display(full_runs[['name', 'base', 'seed', 'train_rows', 'rounds']].sort_values(['base', 'seed']))
full_pred = np.mean([mean_test([FULL_DIR / f'FULL_{b}_seed{s}__test.csv' for s in SEED_TAGS]) for b in CAT_BASE_GROUPS], axis=0)
fold_pred = np.mean([mean_test([CV_DIR / f'{b}{t}__fold{k}__test.csv' for t in SEED_TAGS.values() for k in range(1, 6)])
                     for b in CAT_BASE_GROUPS], axis=0)
checks = []
for label, path, expected in [
    ('最终提交 full_Bseeds', FULL_DIR / 'submission_full_Bseeds.csv', full_pred),
    ('备选 hybrid_50_50', FULL_DIR / 'submission_hybrid_50_50.csv', 0.5 * full_pred + 0.5 * fold_pred),
]:
    saved = pd.read_csv(path, dtype={'id': str})
    require(saved['id'].equals(sample_submission['id']), f'{path.name} 的 ID 顺序与提交模板不同')
    checks.append({'文件': label, '与重新计算的最大差异': float(np.abs(saved['accepted'].to_numpy() - expected).max())})
display(pd.DataFrame(checks))
print(f'全量模型与折平均模型的测试预测相关系数：{np.corrcoef(full_pred, fold_pred)[0, 1]:.5f}')

## 4. 提交与公开榜

公开榜只用约 30% 的测试数据（≈ 13,500 行），0.001 量级的差异基本是噪声（见 05），所以选模以五折和学习曲线为准；下面的公开榜分数顺序与它们的预测一致。

In [ ]:
public = pd.DataFrame([
    ('CPU CatBoost R2, 56 raw features', 0.91604, 0.91857),
    ('GPU CatBoost R2 (same parameters)', 0.91498, 0.91764),
    ('C11 + round-1 features (M0)', 0.91931, 0.92073),
    ('M1+M2+M3 (seed 42), fold average', 0.92051, 0.92190),
    ('B_seeds: M1/M2/M3 x 3 seeds, fold average', 0.92073, 0.92197),
    ('hybrid: 50% full-data + 50% fold average', np.nan, 0.92259),
    ('full_Bseeds: 9 models on all 105k rows (FINAL)', np.nan, 0.92301),
], columns=['submission', '5-fold AUC', 'public LB AUC'])
display(public)

**最终提交**：`submission_full_Bseeds.csv`（主选，公开榜 0.92301）与 `submission_hybrid_50_50.csv`（备选，0.92259）；副本在项目根目录的 `submissions/`。